In [9]:
import ollama
import json
import re

# -----------------------------
# 1. CALL LLM
# -----------------------------
def call_llm(user_input):
    prompt = f"""
You are a system dynamics expert. Your job is to extract stock-and-flow structures from text descriptions.

Text:
\"\"\"{user_input}\"\"\"

REQUIRED OUTPUT FORMAT - You MUST return valid JSON with ALL of these keys:
{{
  "stocks": ["Name of accumulating quantity"],
  "inflows": ["Name of rate that increases stock"],
  "outflows": ["Name of rate that decreases stock"],
  "auxiliary": ["Name of helper variables"],
  "equations": {{
    "stock_name": "INTEG(inflow_name - outflow_name, initial_value)",
    "inflow_name": "mathematical formula",
    "outflow_name": "mathematical formula",
    "auxiliary_name": "formula"
  }},
  "description": "Brief explanation of the system"
}}
 
RULES:
1. STOCKS accumulate over time (e.g., "Views", "Water Level", "Population")
2. INFLOWS are rates that ADD to stocks (e.g., "New Views", "Filling Rate")
3. OUTFLOWS are rates that SUBTRACT from stocks (e.g., "View Decay", "Leakage")
4. AUXILIARY are helper variables that aren't stocks or flows
5. Each stock must have an INTEG() equation
6. Return ONLY the JSON object, no markdown, no code blocks, no explanations
 
EXAMPLE:
Input: "When a post gets more views, people share it more, bringing in even more views."
Output:
{{
  "stocks": ["Post Views"],
  "inflows": ["New Views from Shares"],
  "outflows": ["View Decay"],
  "auxiliary": ["Share Probability", "Decay Rate"],
  "equations": {{
    "Post Views": "INTEG(New Views from Shares - View Decay, 0)",
    "New Views from Shares": "Post Views * Share Probability",
    "View Decay": "Post Views * Decay Rate",
    "Share Probability": "0.1",
    "Decay Rate": "0.05"
  }},
  "description": "A reinforcing loop where more views lead to more shares, creating exponential growth."

  Now analyze the given text and return JSON only.
}}
"""

    response = ollama.chat(
        model='qwen3.5:2b',
        messages=[{"role": "user", "content": prompt}]
    )

    return response['message']['content']


# -----------------------------
# 2. PARSE JSON SAFELY
# -----------------------------
def parse_json(response_text):
    try:
        # Try direct parsing first
        return json.loads(response_text)
    except json.JSONDecodeError:
        pass
    
    try:
        # Extract JSON from messy output (handles markdown code blocks)
        json_match = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', response_text, re.DOTALL)
        if json_match:
            return json.loads(json_match.group(1))
    except json.JSONDecodeError:
        pass
    
    try:
        # Extract raw JSON object
        json_match = re.search(r'\{.*\}', response_text, re.DOTALL)
        if json_match:
            return json.loads(json_match.group())
    except json.JSONDecodeError as e:
        print("JSON parsing error:", e)
    
    return None

# -----------------------------
# 2B. EXTRACT MISSING FIELDS FROM JSON
# -----------------------------
def extract_missing_fields(data):
    """If stocks/inflows/outflows are missing, try to extract from equations"""
    if not data or not data.get("equations"):
        return data
    
    equations = data["equations"]
    
    # Extract stocks (those with INTEG)
    if not data.get("stocks"):
        stocks = [var for var, eq in equations.items() if "INTEG" in eq.upper()]
        if stocks:
            data["stocks"] = stocks
    
    # Extract inflows (mentioned in INTEG right side before minus)
    if not data.get("inflows"):
        inflows = set()
        for var, eq in equations.items():
            if "INTEG" in eq.upper():
                # Parse "INTEG(inflow - outflow, init)"
                match = re.search(r'INTEG\s*\(\s*([^-]+?)\s*-', eq, re.IGNORECASE)
                if match:
                    inflows.add(match.group(1).strip())
        if inflows:
            data["inflows"] = list(inflows)
    
    # Extract outflows (mentioned in INTEG right side after minus)
    if not data.get("outflows"):
        outflows = set()
        for var, eq in equations.items():
            if "INTEG" in eq.upper():
                # Parse "INTEG(inflow - outflow, init)"
                match = re.search(r'INTEG\s*\([^-]*-\s*([^,]+?)\s*,', eq, re.IGNORECASE)
                if match:
                    outflows.add(match.group(1).strip())
        if outflows:
            data["outflows"] = list(outflows)
    
    return data
 

# -----------------------------
# 3. VALIDATION
# -----------------------------
def validate(data):
    errors = []
 
    if not data:
        errors.append("No data parsed")
        return errors
 
    # Check for required keys (using plural forms to match prompt)
    if not data.get("stocks") or len(data.get("stocks", [])) == 0:
        errors.append("Missing stocks")
 
    if not data.get("inflows") or len(data.get("inflows", [])) == 0:
        errors.append("No inflows defined")
 
    if not data.get("outflows") or len(data.get("outflows", [])) == 0:
        errors.append("No outflows defined")
 
    if not data.get("equations") or len(data.get("equations", {})) == 0:
        errors.append("No equations provided")

    return errors


# -----------------------------
# 4. GENERATE MDL
# -----------------------------
def generate_mdl(data):
    """Generate Vensim-style MDL equations from parsed data"""
    mdl_lines = []
 
    stocks = data.get("stocks", [])
    equations = data.get("equations", {})
 
    # Generate stock equations
    for stock in stocks:
        if stock in equations:
            mdl_lines.append(f"{stock} = {equations[stock]}")
        else:
            # Fallback if equation not explicitly provided
            inflows = " + ".join(data.get("inflows", []))
            outflows = " + ".join(data.get("outflows", []))
            mdl_lines.append(f"{stock} = INTEG({inflows} - {outflows}, 0)")
 
    # Add other equations
    for var, equation in equations.items():
        if var not in stocks:
            mdl_lines.append(f"{var} = {equation}")
 
    return "\n".join(mdl_lines)


# -----------------------------
# 5. FULL PIPELINE
# -----------------------------
def run_pipeline(user_input):
    print("🔹 Input:", user_input)
    print("\n" + "="*70)
 
    raw = call_llm(user_input)
    print("\n🔹 Raw LLM Output:")
    print(raw)
    print("="*70)
 
    parsed = parse_json(raw)
    print("\n🔹 Parsed JSON (before extraction):")
    if parsed:
        print(json.dumps(parsed, indent=2))
    else:
        print("Failed to parse JSON")
    
    # Try to extract missing stocks/inflows/outflows from equations
    parsed = extract_missing_fields(parsed)
    print("\n🔹 After field extraction:")
    if parsed:
        print(json.dumps(parsed, indent=2))
    print("="*70)
 
    errors = validate(parsed)
 
    if errors:
        print("\n❌ Validation Errors:")
        for error in errors:
            print(f"  - {error}")
        return None
 
    mdl = generate_mdl(parsed)
    print("\n✅ MDL Output:")
    print(mdl)
    print("="*70)
 
    return parsed, mdl

In [10]:
run_pipeline("The problem is that popular content spreads out of control. When a post gets more views, people are more likely to share it, which brings in even more views. This loop keeps feeding itself with no natural stop.")

🔹 Input: The problem is that popular content spreads out of control. When a post gets more views, people are more likely to share it, which brings in even more views. This loop keeps feeding itself with no natural stop.


🔹 Raw LLM Output:


🔹 Parsed JSON (before extraction):
Failed to parse JSON

🔹 After field extraction:

❌ Validation Errors:
  - No data parsed
